In [1]:
import numpy as np
import matplotlib.pyplot as plt

$\omega$

| Linear | Rotational | Relationship |
| --- | --- | --- |
|position (x) | Angle (θ) | --- |
|Velocity (v) | Anglar Velocity  ($\omega$) | $\omega$ = d $\theta$/d t |
|Mass m | Moment of inertia $\iota$ | $\iota$ = $\Sigma$ $m_i$ $r_i^2$ |
|Force F | Torque $tau$ | $tau$ = r F sin($\phi$) = r x F |
| Momentum p = mv | Angular momentum L = $\iota$ $\omega$ | --- |
|F = ma | $\tau$ = $\iota$ $\alpha$ | $\alpha$ = d $\omega$ / d t |
| KE = $\frac{1}{2}$ m $v^2$ | K $E_rot$ = $\frac{1}{2}$ $\iota$ $\omega^2$ | --- |

|Shape (axis through center)| Moment of Inertia|
|---|---|
|point mass at radius r | m $r^2$ |
|Solid disk / cylinder | $\frac{1}{2}$ m $R^2$ |
|Hoop / thin ring | m $R^2$ |
|soild sphere | $\frac{2}{5}$ m $R^2$|
|Rod about center | $\frac{1}{12}$ m $L^2$|

In [2]:
def moment_of_inertia(masses, positions, axis):
    """I about a given axis point for a set of point masses."""
    total = 0.0
    for m, r in zip(masses, positions):
        total += m * np.sum((np.asarray(r) - np.asarray(axis))**2)
    return total

In [3]:
# --- Verification of the Parallel-Axis Theorem ---

# 1. Define four equal masses at the corners of a square
# Let's place a square of side length 2 centered at the origin (0, 0)
m_individual = 1.0
masses = [m_individual, m_individual, m_individual, m_individual]

positions = [
    [-1.0, -1.0],  # Bottom-left
    [ 1.0, -1.0],  # Bottom-right
    [ 1.0,  1.0],  # Top-right
    [-1.0,  1.0]   # Top-left
]

# 2. Compute I about the center (0, 0)
center_axis = [0.0, 0.0]
I_center = moment_of_inertia(masses, positions, center_axis)

# 3. Compute I about one corner (e.g., top-right corner at [1.0, 1.0])
corner_axis = [1.0, 1.0]
I_corner = moment_of_inertia(masses, positions, corner_axis)

# 4. Calculate the expected theoretical value using the Parallel-Axis Theorem
# Total mass M
M = sum(masses) 

# Distance d from center (0,0) to a corner (1,1) -> d = sqrt(1^2 + 1^2) = sqrt(2)
# Therefore, d^2 = 2.0
d_squared = np.sum((np.asarray(corner_axis) - np.asarray(center_axis))**2)

I_corner_expected = I_center + M * d_squared

# 5. Print results to confirm they match
print(f"I_center:          {I_center}")
print(f"I_corner (calc):   {I_corner}")
print(f"I_corner (theory): {I_corner_expected}")
print(f"Verification Match: {np.isclose(I_corner, I_corner_expected)}")


I_center:          8.0
I_corner (calc):   16.0
I_corner (theory): 16.0
Verification Match: True


In [4]:
def rotate_step(theta, omega, torque, I, dt):
    """Euler-Cromer step for rotation."""
    alpha = torque / I
    omega = omega + alpha * dt      # angular velocity first
    theta = theta + omega * dt      # then angle (Euler-Cromer)
    return theta, omega

In [5]:
m, g, d = 1.0, 9.81, 0.5
I = (1/3) * m * (2*d)**2    # rod of length 2d about its end

def torque(theta):
    return -m * g * d * np.sin(theta)

theta, omega = 0.5, 0.0   # released from 0.5 rad
dt = 0.001
for _ in range(10000):
    theta, omega = rotate_step(theta, omega, torque(theta), I, dt)

In [6]:
def rolling_accel(theta_incline, c, g=9.81):
    """Linear acceleration down an incline for a rolling body.
    c = I/(m R^2): hoop=1, disk=0.5, sphere=0.4."""
    return g * np.sin(theta_incline) / (1 + c)

**5   Conservation of Angular Momentum as a Diagnostic**